# Графики для статьи в журнал IEEE

Фигуры строятся для двух статейных экспериментов: `exp1` — малый \(\Delta u\)
(2026-03-12), `exp2` — большой \(\Delta u\) (2026-03-26). Каждый график
вызывается **отдельно**, чтобы пределы осей (`ylim_pv` / `ylim_co`) можно было
настраивать под каждый эксперимент.

Фигуры на эксперимент:
1. **PV (скользящее среднее ±СКО) + управляющий сигнал** — по оси времени (мин)
   → `pv_mean_std_and_control_output_exp1/2`;
2. **2×2 гистограммы** распределений приращения управления \(\Delta u\) и ошибки
   для ПИД- и НС-контроллера → `action_and_error_histograms_2x2_exp1/2`.

Данные берутся напрямую из нового API (`exp.interaction` / `exp.params`).
Картинки сохраняются в `output/ieee/` (в git не трекаются).

In [ ]:
import colorsys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nn_laser_stabilizer.analysis.experiments import NeuralControllerExperiment, ExperimentId

In [ ]:
def hsl_to_rgb_normalized(h, s, l):
    # colorsys использует HLS (не HSL), поэтому порядок (h, l, s)
    return colorsys.hls_to_rgb(h / 360, l / 100, s / 100)

BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
GREEN_RGB = hsl_to_rgb_normalized(120, 72, 44)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)
PURPLE_RGB = hsl_to_rgb_normalized(279, 98, 76)

plt.rcParams.update({
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 24,
})

In [ ]:
ADC_REF_VOLTAGE = 1.1
ADC_MAX_COUNT = 1023

DAC_REF_VOLTAGE = 5.0
DAC_MAX_COUNT = 4095

ADC_TO_VOLT = ADC_REF_VOLTAGE / ADC_MAX_COUNT
DAC_TO_VOLT = DAC_REF_VOLTAGE / DAC_MAX_COUNT

In [ ]:
OUTPUT_DIR = Path("output") / "ieee"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    for ext in ("pdf", "svg", "png"):
        fig.savefig(OUTPUT_DIR / f"{name}.{ext}", bbox_inches="tight")

In [ ]:
def plot_pv_co(exp: NeuralControllerExperiment, axes=None, *, window=5000):
    ia = exp.interaction
    t_min = ia.time.minutes
    x_max = float(ia.global_step.max())
    total_min = float(t_min.iloc[-1])

    def to_min(gs):
        return gs / x_max * total_min

    pv = ia.env.process_variable
    co = ia.env.control_output
    setpoint = pd.Series(exp.params.setpoint, index=pv.index)
    rolling_mean = pv.rolling(window).mean()
    rolling_std = pv.rolling(window).std()

    pid_end = ia.phases.pid.end
    exploration_min = to_min(pid_end.steps) if pid_end is not None else None
    eval_ranges_min = [(to_min(a), to_min(b)) for a, b in ia.phases.eval.ranges]

    if axes is None:
        _, axes = plt.subplots(
            2, 1, figsize=(14, 10), sharex=True,
            gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08},
        )
    ax_pv, ax_co = axes

    ax_pv.fill_between(
        t_min, rolling_mean - rolling_std, rolling_mean + rolling_std,
        color=BLUE_RGB, alpha=0.2, label="±std", rasterized=True,
    )
    ax_pv.plot(t_min, rolling_mean, color=BLUE_RGB, linewidth=3)
    ax_pv.plot(t_min, setpoint, linestyle="--", linewidth=5, color=RED_RGB, label="Setpoint")
    if exploration_min:
        ax_pv.axvline(x=exploration_min, color="orange", linestyle="--", linewidth=5,
                      label="Transition to NN control")
    for i, (x0, x1) in enumerate(eval_ranges_min):
        ax_pv.axvspan(x0, x1, color="orange", alpha=0.2,
                      label="Evaluation Mode" if i == 0 else None)
    ax_pv.grid(False)
    ax_pv.set_ylabel("Photodetector Signal")

    ax_co.plot(t_min, co, color=GREEN_RGB, alpha=0.7, linewidth=1)
    if exploration_min:
        ax_co.axvline(x=exploration_min, color="orange", linestyle="--", linewidth=5)
    for x0, x1 in eval_ranges_min:
        ax_co.axvspan(x0, x1, color="orange", alpha=0.2)
    ax_co.grid(False)
    ax_co.set_ylabel("Control Signal")
    ax_co.set_xlabel("Time, min")

    ax_pv.text(-0.1, 1.05, "(a)", transform=ax_pv.transAxes, fontsize=30, va="center", ha="center")
    ax_co.text(-0.1, 1.02, "(b)", transform=ax_co.transAxes, fontsize=30, va="center", ha="center")

    handles, labels = ax_pv.get_legend_handles_labels()
    ax_pv.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.02),
                 ncol=2, frameon=False)
    ax_pv.figure.align_ylabels([ax_pv, ax_co])

    return ax_pv, ax_co

In [ ]:
def plot_pv_co_with_phys_units(exp: NeuralControllerExperiment, axes=None, *, window=5000):
    ia = exp.interaction
    t_min = ia.time.minutes
    x_max = float(ia.global_step.max())
    total_min = float(t_min.iloc[-1])

    def to_min(gs):
        return gs / x_max * total_min

    pv = ia.env.process_variable * ADC_TO_VOLT
    co = ia.env.control_output * DAC_TO_VOLT
    setpoint = pd.Series(exp.params.setpoint * ADC_TO_VOLT, index=pv.index)
    rolling_mean = pv.rolling(window).mean()
    rolling_std = pv.rolling(window).std()

    pid_end = ia.phases.pid.end
    exploration_min = to_min(pid_end.steps) if pid_end is not None else None
    eval_ranges_min = [(to_min(a), to_min(b)) for a, b in ia.phases.eval.ranges]

    if axes is None:
        _, axes = plt.subplots(
            2, 1, figsize=(14, 10), sharex=True,
            gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08},
        )
    ax_pv, ax_co = axes

    ax_pv.fill_between(
        t_min, rolling_mean - rolling_std, rolling_mean + rolling_std,
        color=BLUE_RGB, alpha=0.2, label="±std", rasterized=True,
    )
    ax_pv.plot(t_min, rolling_mean, color=BLUE_RGB, linewidth=3)
    ax_pv.plot(t_min, setpoint, linestyle="--", linewidth=5, color=RED_RGB, label="Setpoint")
    if exploration_min:
        ax_pv.axvline(x=exploration_min, color="orange", linestyle="--", linewidth=5,
                      label="Transition to NN control")
    for i, (x0, x1) in enumerate(eval_ranges_min):
        ax_pv.axvspan(x0, x1, color="orange", alpha=0.2,
                      label="Evaluation Mode" if i == 0 else None)
    ax_pv.grid(False)
    ax_pv.set_ylabel("Photodetector Signal, V")

    ax_co.plot(t_min, co, color=GREEN_RGB, alpha=0.7, linewidth=1)
    if exploration_min:
        ax_co.axvline(x=exploration_min, color="orange", linestyle="--", linewidth=5)
    for x0, x1 in eval_ranges_min:
        ax_co.axvspan(x0, x1, color="orange", alpha=0.2)
    ax_co.grid(False)
    ax_co.set_ylabel("Control Signal, V")
    ax_co.set_xlabel("Time, min")

    ax_pv.text(-0.1, 1.05, "(a)", transform=ax_pv.transAxes, fontsize=30, va="center", ha="center")
    ax_co.text(-0.1, 1.02, "(b)", transform=ax_co.transAxes, fontsize=30, va="center", ha="center")

    handles, labels = ax_pv.get_legend_handles_labels()
    ax_pv.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.02),
                 ncol=2, frameon=False)
    ax_pv.figure.align_ylabels([ax_pv, ax_co])

    return ax_pv, ax_co

In [ ]:
def plot_action_error_hist(exp: NeuralControllerExperiment, axes=None, *, tail=100_000):
    ia = exp.interaction
    pid_mask = ia.phases.pid.mask
    nn_mask = ia.phases.nn.mask

    pid_delta_u = ia.actions.control_output_delta[pid_mask]
    nn_delta_u = ia.actions.control_output_delta[nn_mask].iloc[-tail:]
    pid_error = ia.observations.error[pid_mask]
    nn_error = ia.observations.error[nn_mask].iloc[-tail:]

    if len(pid_delta_u) != len(nn_delta_u):
        raise ValueError(
            f"[{exp.id.name}] несопоставимые сегменты для гистограмм: фаза ПИД = "
            f"{len(pid_delta_u)} точек, хвост НС = {len(nn_delta_u)} точек (tail={tail}). "
        )

    if axes is None:
        _, axes = plt.subplots(2, 2, figsize=(14, 10), sharey="row")

    axes[0, 0].hist(pid_delta_u, bins=50, color=PURPLE_RGB, alpha=0.5,
                    edgecolor="black", linewidth=1.5)
    axes[0, 0].set_title("PID controller")
    axes[0, 0].set_xlabel(r"Control increment $\Delta u$")
    axes[0, 0].set_ylabel("Count")

    axes[0, 1].hist(nn_delta_u, bins=50, color=PURPLE_RGB, alpha=0.5,
                    edgecolor="black", linewidth=1.5)
    axes[0, 1].set_title("NN controller")
    axes[0, 1].set_xlabel(r"Control increment $\Delta u$")

    bin_width = 2
    bins = np.arange(
        pid_error.min() - bin_width / 2,
        pid_error.max() + bin_width,
        bin_width,
    )

    axes[1, 0].hist(pid_error, bins=bins, color=BLUE_RGB, alpha=0.5,
                    edgecolor="black", linewidth=1.5)
    axes[1, 0].set_xlabel(r"Error $e$")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].axvline(0, color="black", linestyle="-", linewidth=3)

    axes[1, 1].hist(nn_error, bins=bins, color=BLUE_RGB, alpha=0.5,
                    edgecolor="black", linewidth=1.5)
    axes[1, 1].set_xlabel(r"Error $e$")
    axes[1, 1].axvline(0, color="black", linestyle="-", linewidth=3)

    for ax in axes.ravel():
        ax.set_box_aspect(0.8)
        ax.grid(False)
    for ax, label in zip(axes.ravel(), ["(a)", "(b)", "(c)", "(d)"]):
        ax.text(-0.2, 1.1, label, transform=ax.transAxes, fontsize=30, va="top", ha="left")

    return axes

In [ ]:
exp1 = NeuralControllerExperiment(ExperimentId.from_dir_name("2026-03-12_15-39-27_neural_controller-v3")) 

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08})
ax_pv, ax_co = plot_pv_co(exp1, axes)
fig.tight_layout()
save(fig, "pv_mean_std_and_control_output_exp1")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey="row")
plot_action_error_hist(exp1, axes)
axes[0, 0].set_xlim(-20, 20); axes[0, 1].set_xlim(-20, 20)
axes[1, 0].set_xlim(-60, 15); axes[1, 1].set_xlim(-60, 15)
fig.tight_layout()
save(fig, "action_and_error_histograms_2x2_exp1")

In [ ]:
exp2 = NeuralControllerExperiment(ExperimentId.from_dir_name("2026-03-26_16-54-41_neural_controller-v3"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08})
ax_pv, ax_co = plot_pv_co(exp2, axes)
fig.tight_layout()
save(fig, "pv_mean_std_and_control_output_exp2")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey="row")
plot_action_error_hist(exp2, axes)
axes[0, 0].set_xlim(-55, 55); axes[0, 1].set_xlim(-55, 55)
axes[1, 0].set_xlim(-20, 20); axes[1, 1].set_xlim(-20, 20)
fig.tight_layout()
save(fig, "action_and_error_histograms_2x2_exp2")

## Графики к эксперименту по долгому запуску ПИД-регулятора

In [ ]:
from nn_laser_stabilizer.utils.paths import get_data_dir

import pandas as pd

In [ ]:
df = pd.read_csv(
    get_data_dir() / "250626 PID стабилизация.csv",
    skiprows=[1],      # строка с единицами измерения
    dtype=str
)

df = df.rename(columns={
    "Время": "time",
    "Uфот": "process_variable",
    "setpoint": "setpoint",
    "Uфазовр": "control_output",
})

df["time"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S.%f",
    errors="coerce",
)

for col in ["process_variable", "setpoint", "control_output"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# есть несколько некорректных строк
df = df.dropna(
    subset=[
        "time",
        "process_variable",
        "setpoint",
        "control_output",
    ]
).reset_index(drop=True)

# Устранение нормировки при передаче
df["process_variable"] /= 10
df["setpoint"] /= 10

# В начале запуска уставка равна нулю
mask = (df["setpoint"] != 0) & (df["control_output"] != 0)
df = df.iloc[mask.idxmax():].reset_index(drop=True)

df["time_min"] = (
    df["time"] - df["time"].iloc[0]
).dt.total_seconds() / 60

df = df[[
    "time_min",
    "process_variable",
    "setpoint",
    "control_output",
]]

Перевод отчетов в вольты.

In [ ]:
df["process_variable"] *= ADC_TO_VOLT
df["setpoint"] *= ADC_TO_VOLT
df["control_output"] *= DAC_TO_VOLT

In [ ]:
t_sec = df["time_min"]
pv = df["process_variable"]
co = df["control_output"]
setpoint = df["setpoint"]

fig_co, ax_co = plt.subplots(figsize=(14, 5))

ax_co.plot(
    t_sec,
    co,
    color=GREEN_RGB,
    linewidth=1,
    alpha=0.7,
)

ax_co.set_xlabel("Time, min")
ax_co.set_xlim(left=-1)

ax_co.set_ylabel("Control Signal, V")
ax_co.set_ylim(0, 5)

ax_co_position = ax_co.get_position()

save(fig_co, "control_signal_pid")

In [ ]:
fig_pv, ax_pv = plt.subplots(figsize=(14, 5))

window = 5000
rolling_mean = pv.rolling(window, center=True).mean()
rolling_std = pv.rolling(window, center=True).std()

ax_pv.fill_between(
    t_sec,
    rolling_mean - rolling_std,
    rolling_mean + rolling_std,
    color=BLUE_RGB,
    alpha=0.2,
    label="±std",
    rasterized=True,
)

ax_pv.plot(
    t_sec,
    rolling_mean,
    color=BLUE_RGB,
    linewidth=3,
)

ax_pv.plot(
    t_sec,
    setpoint,
    "--",
    color=RED_RGB,
    linewidth=5,
    label="Setpoint",
)

ax_pv.set_xlabel("Time, min")
ax_pv.set_xlim(left=-1)

ax_pv.set_ylabel("Photodetector Signal, V")
ax_pv.set_ylim(101 * ADC_TO_VOLT, 151 * ADC_TO_VOLT)

ax_pv.legend(
    loc="upper center",
    ncol=2,
    frameon=False,
)

ax_pv.set_position(ax_co_position)

save(fig_pv, "photodetector_signal_pid")

## Графики по эксперименту с Cold Start

In [ ]:
exp_cold_start = NeuralControllerExperiment(ExperimentId.from_dir_name("2026-06-10_19-15-43_neural-controller-v3_inference"))

In [ ]:
ia = exp_cold_start.interaction

t_sec = ia.time.minutes
pv = ia.env.process_variable * ADC_TO_VOLT
co = ia.env.control_output * DAC_TO_VOLT
setpoint = pd.Series(
    exp_cold_start.params.setpoint * ADC_TO_VOLT,
    index=pv.index,
)

In [ ]:
fig_co, ax_co = plt.subplots(figsize=(14, 5))

ax_co.plot(
    t_sec,
    co,
    color=GREEN_RGB,
    linewidth=1,
    alpha=0.7,
)

ax_co.set_xlabel("Time, min")
ax_co.set_xlim(left=-1)

ax_co.set_ylabel("Control Signal, V")
ax_co.set_ylim(0, 5)

ax_co_position = ax_co.get_position()

save(fig_co, "control_signal_cold_start")

In [ ]:
fig_pv, ax_pv = plt.subplots(figsize=(14, 5))

window = 5000
rolling_mean = pv.rolling(window, center=True).mean()
rolling_std = pv.rolling(window, center=True).std()

ax_pv.fill_between(
    t_sec,
    rolling_mean - rolling_std,
    rolling_mean + rolling_std,
    color=BLUE_RGB,
    alpha=0.2,
    label="±std",
    rasterized=True,
)

ax_pv.plot(
    t_sec,
    rolling_mean,
    color=BLUE_RGB,
    linewidth=3,
)

ax_pv.plot(
    t_sec,
    setpoint,
    "--",
    color=RED_RGB,
    linewidth=5,
    label="Setpoint",
)

ax_pv.set_xlabel("Time, min")
ax_pv.set_xlim(left=-1)

ax_pv.set_ylabel("Photodetector Signal, V")
ax_pv.set_ylim(101 * ADC_TO_VOLT, 151 * ADC_TO_VOLT)

ax_pv.legend(
    loc="upper center",
    ncol=2,
    frameon=False,
)

ax_pv.set_position(ax_co_position)

save(fig_pv, "photodetector_signal_cold_start")

## График для ответа на вопрос о времени достижения режима

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08})
window = 5000
ia = exp_cold_start.interaction
t_sec = ia.time.seconds
x_max = float(ia.global_step.max())
total_min = float(t_sec.iloc[-1])

def to_min(gs):
    return gs / x_max * total_min

pv = ia.env.process_variable * ADC_TO_VOLT
co = ia.env.control_output * DAC_TO_VOLT
setpoint = pd.Series(exp_cold_start.params.setpoint * ADC_TO_VOLT, index=pv.index)

pid_end = ia.phases.pid.end
exploration_min = to_min(pid_end.steps) if pid_end is not None else None
eval_ranges_min = [(to_min(a), to_min(b)) for a, b in ia.phases.eval.ranges]

if axes is None:
    _, axes = plt.subplots(
        2, 1, figsize=(14, 10), sharex=True,
        gridspec_kw={"height_ratios": [1, 1], "hspace": 0.08},
    )
ax_pv, ax_co = axes

ax_pv.plot(t_sec, pv, color=BLUE_RGB, linewidth=3)
ax_pv.plot(t_sec, setpoint, linestyle="--", linewidth=5, color=RED_RGB, label="Setpoint")
ax_pv.grid(False)
ax_pv.set_ylabel("Photodetector Signal, V")

ax_co.plot(t_sec, co, color=GREEN_RGB, alpha=0.7, linewidth=1)
ax_co.grid(False)
ax_co.set_ylabel("Control Signal, V")
ax_co.set_xlabel("Time, sec")

ax_pv.text(-0.1, 1.08, "(a)", transform=ax_pv.transAxes, fontsize=30, va="center", ha="center")
ax_co.text(-0.1, 0.96, "(b)", transform=ax_co.transAxes, fontsize=30, va="center", ha="center")

handles, labels = ax_pv.get_legend_handles_labels()
ax_pv.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.02),
                ncol=2, frameon=False)
ax_pv.figure.align_ylabels([ax_pv, ax_co])

ax_pv.set_xlim(0, 6)
ax_co.set_xlim(0, 6)

fig.tight_layout()
save(fig, "pv_mean_std_and_control_output_cold_start")

## График для статьи по сравнением устойчивости к воздействиям

In [ ]:
df = pd.read_csv(
    get_data_dir() / "PID реакция на удар.csv",
    skiprows=[1],      # строка с единицами измерения
    dtype=str
)

df = df.rename(columns={
    "Время": "time_sec",
    "Uфот": "process_variable",
    "setpoint": "setpoint",
    "Uфазовр": "control_output",
})

for col in ["process_variable", "setpoint", "control_output", "time_sec"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[
        "time_sec",
        "process_variable",
        "setpoint",
        "control_output",
    ]
).reset_index(drop=True)

df["process_variable"] /= 10
df["setpoint"] /= 10

df = df[[
    "time_sec",
    "process_variable",
    "setpoint",
    "control_output",
]]

Оставим только часть с `setpoint == 110`. 

In [ ]:
df_110 = df[df["setpoint"] == 110].copy()

df_110["time_sec"] -= df_110["time_sec"].iloc[0]

In [ ]:
df_110["process_variable"] *= ADC_TO_VOLT
df_110["setpoint"] *= ADC_TO_VOLT
df_110["control_output"] *= DAC_TO_VOLT

In [ ]:
t_sec = df_110["time_sec"]
pv = df_110["process_variable"]
co = df_110["control_output"]
setpoint = df_110["setpoint"]

In [ ]:
fig_co, ax_co = plt.subplots(figsize=(14, 5))

ax_co.plot(
    t_sec,
    co,
    color=GREEN_RGB,
    linewidth=1,
    alpha=0.7,
)

ax_co.set_xlabel("Time, sec")
ax_co.set_xlim(left=-1)

ax_co.set_ylabel("Control Signal, V")
ax_co.set_ylim(0, 5)

ax_co_position = ax_co.get_position()

save(fig_co, "control_signal_pid_disturbance")

In [ ]:
fig_pv, ax_pv = plt.subplots(figsize=(14, 5))

ax_pv.plot(
    t_sec,
    pv,
    color=BLUE_RGB,
    linewidth=3,
)

ax_pv.plot(
    t_sec,
    setpoint,
    "--",
    color=RED_RGB,
    linewidth=5,
    label="Setpoint",
)

ax_pv.set_xlabel("Time, sec")
ax_pv.set_xlim(left=-1)

ax_pv.set_ylabel("Photodetector Signal, V")

ax_pv.legend(
    loc="upper center",
    ncol=2,
    frameon=False,
)

ax_pv.set_position(ax_co_position)

save(fig_pv, "photodetector_signal_pid_disturbance")

In [ ]:
exp_nn_disturbance = NeuralControllerExperiment(ExperimentId.from_dir_name("2026-06-18_15-36-39_neural-controller-v3_inference"))

In [ ]:
t_sec = exp_nn_disturbance.interaction.time.seconds
pv = exp_nn_disturbance.interaction.env.process_variable * ADC_TO_VOLT
co = exp_nn_disturbance.interaction.env.control_output * DAC_TO_VOLT
setpoint = pd.Series(
    exp_nn_disturbance.params.setpoint * ADC_TO_VOLT,
    index=pv.index,
)

In [ ]:
t0 = 15 * 60
t1 = t0 + 160

mask = (t_sec >= t0) & (t_sec <= t1)

t_sec = t_sec[mask]
pv = pv[mask]
co = co[mask]
setpoint = setpoint[mask]

t_sec = t_sec - t_sec.iloc[0]

In [ ]:
fig_co, ax_co = plt.subplots(figsize=(14, 5))

ax_co.plot(
    t_sec,
    co,
    color=GREEN_RGB,
    linewidth=1,
    alpha=0.7,
)

ax_co.set_xlabel("Time, sec")
ax_co.set_xlim(left=-1)

ax_co.set_ylabel("Control Signal, V")
ax_co.set_ylim(0, 5)

ax_co_position = ax_co.get_position()

save(fig_co, "control_signal_nn_disturbance")

In [ ]:
fig_pv, ax_pv = plt.subplots(figsize=(14, 5))

ax_pv.plot(
    t_sec,
    pv,
    color=BLUE_RGB,
    linewidth=3,
)

ax_pv.plot(
    t_sec,
    setpoint,
    "--",
    color=RED_RGB,
    linewidth=5,
    label="Setpoint",
)

ax_pv.set_xlabel("Time, sec")
ax_pv.set_ylim(top=0.196, bottom=0.095)
ax_pv.set_xlim(left=-1)

ax_pv.set_ylabel("Photodetector Signal, V")

ax_pv.legend(
    loc="upper center",
    ncol=2,
    frameon=False,
)

ax_pv.set_position(ax_co_position)

save(fig_pv, "photodetector_signal_nn_disturbance")